# Tally v2 — disconnect-resilient notebook

This is the working notebook for the Lexsi Tally project (fine-tuning Tiny Aya to extract fields from forest carbon project documents).

**After any runtime disconnect, do this:**
1. Check the runtime is a T4 GPU: Runtime → Change runtime type.
2. Run the **Setup** cell below. It remounts Drive, reinstalls the packages (wheels are cached, so this is faster the second time), and reloads the model from your Drive cache instead of downloading it again.
3. Then run only the later cells whose work isn't already saved in Drive. Helpers skip work when the output file already exists.

The original `tally.ipynb` is left unchanged.


## 1. Setup (run this one after any disconnect)

This one cell does everything needed to get back to a ready model:

- mounts Google Drive
- points Hugging Face at your Drive cache (`HF_HOME`), so Tiny Aya's ~7 GB stays there
- points pip at a Drive cache too, so package *wheels* are reused
- installs CuratorKIT, AlignTune, and AuditKIT
- logs in with your `HF_TOKEN` Colab secret
- loads Tiny Aya in float16 on the GPU, unless it is already in memory

We **do not** install packages with `pip --target` into Drive. That can shadow Colab's preinstalled `torch` and break the GPU stack. Caching the wheels is the safe way to make reinstalls fast.


In [ ]:
from google.colab import drive, userdata
import os

drive.mount("/content/drive")

# Keep the model cache on Drive so it survives disconnects.
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

# Cache pip wheels on Drive so the next !pip install is mostly a local copy.
# Do NOT use pip --target into Drive: that folder would sit on sys.path and
# can shadow Colab's preinstalled torch, which breaks GPU support.
os.environ["PIP_CACHE_DIR"] = "/content/drive/MyDrive/tally/pip_cache"
os.makedirs(os.environ["PIP_CACHE_DIR"], exist_ok=True)

!pip install -q "curatorkit[all]" aligntune "auditkit[transformers]"

import torch
from huggingface_hub import login

login(userdata.get("HF_TOKEN"))

import curatorkit, aligntune, auditkit
from transformers import AutoTokenizer, AutoModelForCausalLM

MID = "CohereLabs/tiny-aya-global"

# Re-running this cell after a disconnect is cheap if the model is still in RAM.
# A full disconnect clears RAM, so the model will load from the Drive cache.
if "model" not in globals():
    tok = AutoTokenizer.from_pretrained(MID)
    model = AutoModelForCausalLM.from_pretrained(
        MID,
        dtype=torch.float16,  # not the older torch_dtype=
        device_map="auto",
    )

print("GPU:", torch.cuda.get_device_name(0))
print("ready")


## 2. Project folders

All Tally files live under `/content/drive/MyDrive/tally`, which is the `tally` folder in My Drive. Subfolders:

- `pdds` — project design / monitoring / verification PDFs
- `text` — extracted text from those PDFs
- `labels` — structured JSON we'll create later
- `evals` — scoring results from AuditKIT
- `checkpoints` — LoRA weights saved during fine-tuning

`exist_ok=True` means this cell is safe to re-run; it won't fail if the folders are already there.


In [ ]:
import os

TALLY = "/content/drive/MyDrive/tally"

for name in ("pdds", "text", "labels", "evals", "checkpoints"):
    os.makedirs(os.path.join(TALLY, name), exist_ok=True)

print("TALLY =", TALLY)
print("folders:", ", ".join(sorted(os.listdir(TALLY))))


## 3. Helpers

Two small tools we'll reuse in later cells:

- `cached_json(path, fn)` and `cached_text(path, fn)` — if the file already exists, read and return it. If not, call `fn()`, save the result, and return it. After a disconnect you don't redo finished work.
- `ask(prompt, max_new_tokens=200, temperature=None)` — send a chat message to Tiny Aya and return only the reply text. Greedy decoding is the default (same answer each time, which is what we want for extraction). Pass a `temperature` to sample instead.

In current Transformers, `apply_chat_template(..., return_tensors="pt")` returns a `BatchEncoding` (a dict-like object with `.input_ids` and `.attention_mask`), not a bare tensor. `ask` uses those fields so generation works.


In [ ]:
import json
from pathlib import Path


def cached_json(path, fn):
    path = Path(path)
    if path.exists():
        with path.open(encoding="utf-8") as f:
            return json.load(f)
    data = fn()
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    return data


def cached_text(path, fn):
    path = Path(path)
    if path.exists():
        return path.read_text(encoding="utf-8")
    text = fn()
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
    return text


def ask(prompt, max_new_tokens=200, temperature=None):
    messages = [{"role": "user", "content": prompt}]
    inputs = tok.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    gen_kwargs = {
        "input_ids": inputs.input_ids,
        "attention_mask": inputs.attention_mask,
        "max_new_tokens": max_new_tokens,
    }
    if temperature is None:
        gen_kwargs["do_sample"] = False  # greedy: pick the highest-scoring token
    else:
        gen_kwargs["do_sample"] = True
        gen_kwargs["temperature"] = temperature

    output = model.generate(**gen_kwargs)
    # Slice off the prompt tokens so we only decode the model's reply.
    reply_ids = output[0][inputs.input_ids.shape[1]:]
    return tok.decode(reply_ids, skip_special_tokens=True)


## 4. Smoke test

Ask the model for a tiny JSON object, then try `json.loads` on the reply. This is a quick check that (a) generation works and (b) the base model can already produce structured output — the skill Tally will train it to do well on project documents.


In [ ]:
import json

reply = ask('Reply with JSON: {"status": "ok"}', max_new_tokens=30)
print("Model reply:")
print(reply)
print()

try:
    parsed = json.loads(reply)
    print("Parsed OK:", parsed)
except json.JSONDecodeError as err:
    print("Did not parse as JSON:", err)


## 5. Anatomy

A first look inside Tiny Aya:

- `print(model)` shows the layer tree: the embedding table, 36 decoder layers (each with attention and an MLP), then the output head.
- `model.config` shows the numbers behind it (layer count, hidden size, attention heads, vocabulary size, and the 8,192-token context window).
- The parameter count and the first few weight shapes show the actual matrices.


In [ ]:
print(model)
print(model.config)
total = sum(p.numel() for p in model.parameters())
print(f"parameters: {total/1e9:.2f} B")
for name, p in list(model.named_parameters())[:12]:
    print(name, tuple(p.shape))


## 6. PDF to text, cached

Read every PDF in `TALLY/pdds`, extract the text of each page, and save it to `TALLY/text/<name>.txt`. If that `.txt` already exists, we skip the extraction (the `cached_text` helper).

For each file we also print:

- the page count
- an approximate token count, using Tiny Aya's own tokenizer on the full text

**Important:** Tiny Aya can only attend to **8,192 tokens** at a time (`max_position_embeddings`). Many of these project documents are far longer than that, so later we'll split each document into sections and extract fields from the relevant ones, rather than stuffing a whole PDD into one prompt.


In [ ]:
!pip install -q pymupdf

import fitz  # PyMuPDF
from pathlib import Path

pdd_dir = Path(TALLY) / "pdds"
text_dir = Path(TALLY) / "text"
text_dir.mkdir(parents=True, exist_ok=True)

pdfs = sorted(pdd_dir.glob("*.pdf"))
print(f"Found {len(pdfs)} PDFs in {pdd_dir}")

for pdf in pdfs:
    out_path = text_dir / f"{pdf.stem}.txt"

    def extract(pdf_path=pdf):
        doc = fitz.open(pdf_path)
        pages = []
        for i, page in enumerate(doc, start=1):
            pages.append(f"--- page {i} ---\n{page.get_text()}")
        doc.close()
        return "\n\n".join(pages)

    text = cached_text(out_path, extract)
    page_count = text.count("--- page ")
    n_tokens = len(tok.encode(text))
    over = "  (over the 8,192-token window)" if n_tokens > 8192 else ""
    print(f"{pdf.name}: {page_count} pages, ~{n_tokens:,} tokens{over} -> {out_path.name}")
